# Portfolio Compass: Application Thought Process

**A reproducible project narrative from plan to working Streamlit application**

This notebook explains what was built, why the architecture looks the way it does, which data and prompts shaped it, how the implementation evolved, and what the workflow taught us. It is based on `plan.md`, the current repository, and the feedback/corrections captured during the build.

> **Recommended Markdown table of contents pattern:** use a numbered hierarchy that mirrors the notebook headings (`1`, `1.1`, `1.2`), with Markdown links to each section. Keep the top-level sections short, and use subsection links only where they help navigation. This makes the narrative scannable without turning the table of contents into a second essay.

## Table of contents

1. [Project brief and success criteria](#1-project-brief-and-success-criteria)
   1. [What we built](#11-what-we-built)
   2. [Scope boundaries](#12-scope-boundaries)
2. [Architecture and implementation flow](#2-architecture-and-implementation-flow)
   1. [Module map](#21-module-map)
   2. [End-to-end user flow](#22-end-to-end-user-flow)
3. [Datasets and data contracts](#3-datasets-and-data-contracts)
   1. [Committed fixtures](#31-committed-fixtures)
   2. [Market and benchmark data](#32-market-and-benchmark-data)
4. [Accounting and analytics reasoning](#4-accounting-and-analytics-reasoning)
5. [Vibe-coding prompts and iteration history](#5-vibe-coding-prompts-and-iteration-history)
6. [Testing and evidence](#6-testing-and-evidence)
7. [Learnings and observations](#7-learnings-and-observations)
8. [Limitations and next steps](#8-limitations-and-next-steps)
9. [Embedded application code](#9-embedded-application-code)


## 1. Project brief and success criteria

Portfolio Compass is a plain-language portfolio analytics application for a person who wants to understand what they own, how it is performing, and why. The design in `plan.md` prioritizes explainable calculations, reviewable assumptions, predictable failures, and a polished Streamlit experience.

### 1.1 What we built

The current repository implements a local-first Streamlit application with:

- CSV import with flexible column aliases and normalized transactions.
- Instrument resolution for tickers, common names, aliases, and partial-name suggestions.
- FIFO accounting for buys, sells, fees, supported splits, realized gains, unrealized gains, and income.
- Historical valuation using available closing prices and explicit missing-data warnings.
- Performance measures including XIRR and a daily cash-flow-adjusted approximation.
- Optional S&P 500 comparison through SPY as a clearly labeled ETF proxy.
- A bounded Ask Compass experience that uses portfolio evidence and refuses recommendations, forecasts, and unrelated questions.
- A synthetic sample-data generator and committed CSV fixtures.
- Tests for import validation, sample generation, and XIRR.

### 1.2 Scope boundaries

The application is intentionally one portfolio and one CSV per session. It supports long-only positions, fractional shares, USD-denominated US-traded instruments, and latest completed daily closes. It does not infer brokerage cash, deposits, withdrawals, reinvestment, tax reporting, real-time prices, or complex corporate actions. Those limits are part of the product contract because silently inventing financial facts would be worse than showing an incomplete result.


## 2. Architecture and implementation flow

The implementation follows a layered design so that financial logic remains testable outside Streamlit:

1. **Models and configuration** define normalized transactions, instruments, market data, analysis results, settings, and issue severity.
2. **Importing and instruments** turn messy CSV inputs into reviewable, canonical identities.
3. **Accounting** replays transactions through FIFO lots and reconciles recorded versus estimated income.
4. **Market data** supplies prices, distributions, splits, exchange sessions, and quality flags behind an adapter boundary.
5. **Performance** calculates XIRR, period metrics, daily returns, and benchmark comparison.
6. **AI** routes questions through a strict allowlist and constructs minimal evidence context.
7. **UI** composes the four user-facing screens, charts, downloads, status strips, and empty/error states.

### 2.1 Module map

| Layer | Main files | Responsibility |
|---|---|---|
| Entry point | `app.py` | Starts Streamlit and calls `main()` |
| Domain models | `portfolio_compass/models.py` | Typed records and analysis containers |
| Import | `portfolio_compass/importing.py` | CSV detection, mapping, normalization, validation |
| Identity | `portfolio_compass/instruments.py` | Aliases, canonical IDs, display labels |
| Accounting | `portfolio_compass/accounting.py` | FIFO lots, fees, gains, income |
| Market | `portfolio_compass/market_data.py` | yfinance-backed prices and actions |
| Performance | `portfolio_compass/performance.py` | XIRR, period metrics, SPY comparison |
| AI | `portfolio_compass/ai.py` | Bounded routing, evidence, Groq integration |
| UI | `portfolio_compass/ui.py` | Streamlit screens and interaction flow |
| Fixtures | `Sample Data - Sheet1.csv`, canonical CSVs | Reproducible sample inputs |
| Generator | `sample_data_generator.py` and package module | Synthetic transaction CSV generation |


### 2.2 End-to-end user flow

```text
Upload or explore sample
        ↓
Detect file and suggest column mappings
        ↓
Normalize dates, numbers, transaction types, and identities
        ↓
Review ambiguous names/tickers and validation issues
        ↓
Confirm the transaction ledger
        ↓
Fetch or use labeled market data
        ↓
Replay FIFO accounting + income reconciliation
        ↓
Render overview, performance, holdings, exports, and Ask Compass
```

The key design decision is the review checkpoint before analysis. It makes ambiguity visible and gives the user a chance to correct identity or mapping decisions before those choices affect cost basis, valuation, returns, and AI context.


## 3. Datasets and data contracts

The application uses several data categories, each with a different trust level. Keeping these categories separate was central to making the results explainable.

### 3.1 Committed fixtures

- **`Sample Data - Sheet1.csv`** is the original 13-trade sample and is preserved unchanged. The transaction-only holdings oracle is AAPL 10, MSFT 9, and TSLA 4 units.
- **`portfolio_compass_canonical_1.csv` and `portfolio_compass_canonical_2.csv`** exercise canonical transaction shapes and validation paths.
- **Synthetic generator outputs** are test fixtures only. They use online instrument identity/reference prices when available, but generated prices and trades do not claim to be historical market activity.

### 3.2 Market and benchmark data

- **yfinance** is the real market-data provider path for completed daily closes, distributions, and splits.
- **SPY** is used as an S&P 500 ETF proxy, and the UI labels it as such.
- **Exchange sessions** are used to flag non-trading-day transaction dates and to choose prior closes without looking into the future.
- **Groq** is optional and bounded. It receives only the minimal evidence required for an in-scope question; it does not supply displayed financial facts.

This separation supports graceful degradation: transaction-only views can remain useful when price data is unavailable, while the UI discloses when a complete valuation or return is not possible.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'plan.md').exists() and (ROOT.parent / 'plan.md').exists():
    ROOT = ROOT.parent
print('Notebook root:', ROOT)
sample_path = ROOT / 'Sample Data - Sheet1.csv'
sample = pd.read_csv(sample_path)
print('Sample shape:', sample.shape)
display(sample)


## 4. Accounting and analytics reasoning

The financial calculations are designed around conservation and replay rather than shortcuts. Each security has FIFO lots containing quantity and cost. A buy creates a lot whose cost includes purchase fees. A sell consumes the oldest remaining lots and allocates partial-lot cost proportionally. Sale fees reduce proceeds.

The main identity used for review is:

```text
Total profit = realized gain + unrealized gain + income
```

Historical reporting replays the complete transaction history up to a selected date. A reporting window changes what is shown, but it cannot discard earlier purchases needed to compute FIFO cost basis. Distribution estimates use eligible units before the event and are labeled as estimates. Uploaded distributions can override matched estimates after reconciliation.

The implementation also treats missing data as unknown rather than zero. A missing quote can interrupt a complete valuation, and the resulting warning is carried into the analysis and export context. This is a small but important correctness choice: zero would create false precision.


## 5. Vibe-coding prompts and iteration history

The prompts were most effective when they specified behavior, evidence, and failure handling together. The working pattern was:

1. **Plan first:** define users, scope, data contract, accounting definitions, AI boundaries, deployment assumptions, and acceptance checks in `plan.md`.
2. **Build in layers:** implement import and identity resolution before accounting; accounting before performance; performance before UI; UI before bounded AI.
3. **Use corrections as requirements:** each feedback cycle tightened an ambiguity or prevented a silent assumption.
4. **Verify with fixtures:** preserve the original sample and add independent canonical inputs rather than using the same implementation to validate itself.

Representative prompts used during the workflow included:

- “Build Portfolio Compass from `plan.md`, preserving the sample CSV and implementing the complete initial-release scope.”
- “Make ticker and common-name resolution explicit: show candidates, require confirmation for ambiguity, and invalidate derived results after remapping.”
- “Keep FIFO arithmetic exact, include fees, and test the profit identity at multiple dates.”
- “Bound Ask Compass so it can answer only from the current portfolio and benchmark evidence; refuse recommendations and unrelated questions.”
- “Treat missing quotes and uncertain income as quality states, not zeros or hidden failures.”
- “Add a sample-data generator that produces valid long-only histories and exercises importer aliases without using the original sample as its universe.”

### Iterations tried

- **Initial planning iteration:** established scope and acceptance criteria before implementation.
- **Accounting hardening:** added Decimal-based ledger arithmetic, FIFO lot consumption, fee allocation, split ordering, and income reconciliation.
- **Identity hardening:** expanded aliases and display labels, separated raw input from canonical security identity, and added review-before-analysis.
- **AI hardening:** moved from open-ended generation toward intent routing, allowlisted vocabulary, minimal evidence, refusal paths, and deterministic templates.
- **Data-quality iteration:** added non-trading-day warnings, prior-close valuation, missing-price states, and explicit estimated-income labels.
- **Generator iteration:** added deterministic synthetic histories, event validity rules, supported input shapes, and a non-stock requirement for multi-investment portfolios.


## 6. Testing and evidence

The repository includes focused tests for the highest-risk seams:

- `tests/test_import_validation.py` checks mapping and validation behavior.
- `tests/test_sample_data_generator.py` checks generated transaction validity and supported input shapes.
- `tests/test_xirr.py` checks return calculations.

The notebook also runs lightweight repository checks below. These are intentionally local and credential-free; live yfinance and Groq calls are not triggered automatically.


In [ ]:
import subprocess, sys

project = ROOT
pytest_cmd = [sys.executable, '-m', 'pytest', '-q']
result = subprocess.run(pytest_cmd, cwd=project, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
print('pytest exit code:', result.returncode)


In [ ]:
# Verify the transaction-only holdings oracle from the project plan.
+from collections import defaultdict
+
+def canonical_kind(value):
+    value = str(value).strip().upper()
+    return {'BUY': 'BUY', 'SELL': 'SELL', 'DISTRIBUTION': 'DISTRIBUTION'}.get(value, value)
+
+holdings = defaultdict(float)
+for _, row in sample.iterrows():
+    kind = canonical_kind(row.get('transaction_type', row.get('Type', '')))
+    qty = float(row.get('quantity', row.get('Quantity', 0)) or 0)
+    symbol = str(row.get('symbol', row.get('Ticker', row.get('instrument_input', '')))).strip().upper()
+    if kind == 'BUY': holdings[symbol] += qty
+    elif kind == 'SELL': holdings[symbol] -= qty
+print(dict(holdings))
+print('Expected oracle: AAPL=10, MSFT=9, TSLA=4')


## 7. Learnings and observations

The workflow surfaced several durable lessons.

- **A financial app needs visible assumptions.** Labels such as “SPY proxy,” “estimated income,” “latest completed close,” and “brokerage cash excluded” are part of correctness, not cosmetic copy.
- **Identity is accounting input.** A guessed ticker can contaminate every downstream metric, so resolution and confirmation belong before analysis.
- **Boundaries improve AI usefulness.** The most reliable assistant behavior came from routing and evidence constraints, not from asking a model to calculate the portfolio freely.
- **Missing is a value state.** Treating unavailable data as zero makes dashboards look complete while reducing trust.
- **Fixtures should be independent.** The original sample is useful as an oracle, but additional fixtures and generator checks are needed to catch implementation-specific errors.
- **UI states are part of the model.** Empty, partial, warning, outage, and unavailable-AI states need explicit representation because they change what conclusions are safe.
- **Feedback became product policy.** Corrections about ambiguity, future prices, uploaded distributions, spreadsheet-safe exports, and refusal behavior were converted into explicit rules instead of informal promises.


## 8. Limitations and next steps

The current application remains local-first. Live provider coverage, Groq model availability, hosting quotas, and Community Cloud deployment must be verified at release time. Complex corporate actions, opening balances, multiple uploads, tax reporting, real-time prices, ETF look-through, advanced risk analytics, and persistent storage remain deferred in the plan.

The next useful engineering steps are to keep expanding independent accounting fixtures, measure capacity against the target transaction/ticker limits, exercise both themes and mobile layouts, and perform separate live-service smoke checks with credentials configured privately.


## 9. Embedded application code

The following cells embed the current project source at notebook-build time. Keeping the source in code cells makes the notebook a reviewable snapshot of the implementation while leaving the actual modules as the runnable application.


### `app.py`

This is the implementation snapshot used by the application.

```python
"""Run with: python -m streamlit run app.py"""

from portfolio_compass.ui import main

main()

```

### `portfolio_compass/__init__.py`

This is the implementation snapshot used by the application.

```python
"""Portfolio Compass: deterministic investment accounting and a scoped analyst."""

```

### `portfolio_compass/accounting.py`

This is the implementation snapshot used by the application.

```python
"""FIFO accounting with exact ledger arithmetic and explicit distribution reconciliation."""

from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import timedelta
from decimal import Decimal

from .models import ZERO, Analysis, Issue


@dataclass
class Lot:
    quantity: Decimal
    cost: Decimal


class Book:
    def __init__(self):
        self.lots = defaultdict(deque)
        self.quantity = defaultdict(lambda: ZERO)
        self.cost = defaultdict(lambda: ZERO)
        self.realized = defaultdict(lambda: ZERO)
        self.purchases = defaultdict(lambda: ZERO)
        self.sales = defaultdict(lambda: ZERO)

    def split(self, symbol, ratio):
        if ratio <= 0 or not ratio.is_finite():
            raise ValueError(f'{symbol}: invalid split ratio.')
        for lot in self.lots[symbol]:
            lot.quantity *= ratio
        self.quantity[symbol] *= ratio

    def trade(self, tx):
        if tx.kind == 'DISTRIBUTION':
            return
        symbol, qty = tx.symbol, tx.quantity
        if tx.kind == 'BUY':
            amount = qty * tx.price + tx.fees
            self.lots[symbol].append(Lot(qty, amount))
            self.quantity[symbol] += qty
            self.cost[symbol] += amount
            self.purchases[symbol] += amount
            return
        if qty > self.quantity[symbol]:
            raise ValueError(f'Row {tx.row}: {symbol} sale exceeds units owned on {tx.day}. Check history, order, or split basis.')
        consumed, remaining = ZERO, qty
        while remaining > 0:
            lot = self.lots[symbol][0]
            take = min(remaining, lot.quantity)
            allocation = lot.cost if take == lot.quantity else lot.cost * take / lot.quantity
            consumed += allocation
            lot.cost -= allocation
            lot.quantity -= take
            remaining -= take
            if not lot.quantity:
                self.lots[symbol].popleft()
        proceeds = qty * tx.price - tx.fees
        self.quantity[symbol] -= qty
        self.cost[symbol] -= consumed
        if not self.quantity[symbol]:
            self.cost[symbol] = ZERO
        self.realized[symbol] += proceeds - consumed
        self.sales[symbol] += proceeds


def estimate_income(transactions, actions, end):
    """Replay split and trade events; dividend entitlement precedes same-day trades."""
    events = defaultdict(list)
    for tx in transactions:
        if tx.day <= end:
            events[tx.day].append((2, tx))
    for action in actions:
        if action.day <= end:
            events[action.day].append((0 if action.kind == 'SPLIT' else 1, action))
    book, result = Book(), []
    for day in sorted(events):
        for priority, event in sorted(events[day], key=lambda x: x[0]):
            if priority == 0:
                book.split(event.symbol, event.value)
            elif priority == 1:
                units = book.quantity[event.symbol]
                if units > 0:
                    result.append({'id': event.id, 'date': day, 'symbol': event.symbol,
                                   'amount': units * event.value, 'units': units,
                                   'source': 'Estimated (ex-date)', 'event': event.id})
            else:
                book.trade(event)
    return result


def reconcile_income(transactions, estimates, decisions=None):
    decisions = decisions or {}
    actuals = [t for t in transactions if t.kind == 'DISTRIBUTION']
    candidates = {tx.id: [e for e in estimates if e['symbol'] == tx.symbol and abs((tx.day - e['date']).days) <= 45] for tx in actuals}
    claimed, conflicts, records = set(), [], []
    for tx in actuals:
        nearby = candidates[tx.id]
        choice = decisions.get(tx.id)
        if choice is None:
            exact = [e for e in nearby if abs(tx.amount - e['amount']) <= Decimal('.02')]
            if len(exact) == 1 and sum(exact[0] in candidates[t.id] for t in actuals) == 1:
                choice = exact[0]['id']
            elif nearby:
                conflicts.append({'id': tx.id, 'row': tx.row, 'symbol': tx.symbol, 'date': tx.day,
                                  'amount': float(tx.amount), 'candidates': nearby})
        if choice and choice != 'independent':
            if choice not in {e['id'] for e in nearby} or choice in claimed:
                raise ValueError('Each uploaded distribution must match at most one unclaimed provider event.')
            claimed.add(choice)
        records.append({'id': tx.id, 'date': tx.day, 'symbol': tx.symbol, 'amount': tx.amount,
                        'units': None, 'source': 'Recorded', 'event': choice or ''})
    records.extend(e for e in estimates if e['id'] not in claimed)
    return sorted(records, key=lambda x: x['date']), conflicts


def analyze(transactions, instruments, market, sessions, decisions=None, estimate=True):
    if not transactions:
        raise ValueError('No transactions to analyze.')
    end = market.as_of
    if end is None or end < transactions[0].day:
        raise ValueError('The valuation date must not precede the first transaction.')
    if transactions[-1].day > end:
        raise ValueError('Transactions after the last completed market session cannot be valued yet.')
    estimates = estimate_income(transactions, market.actions, end) if estimate else []
    income, conflicts = reconcile_income(transactions, estimates, decisions)
    by_day, actions, payouts = defaultdict(list), defaultdict(list), defaultdict(list)
    for tx in transactions:
        by_day[tx.day].append(tx)
    for event in market.actions:
        if event.kind == 'SPLIT':
            actions[event.day].append(event)
    for item in income:
        payouts[item['date']].append(item)
    issues = list(market.issues)
    nontrading = sorted({t.day for t in transactions if t.day not in sessions})
    if nontrading:
        issues.append(Issue('warning', f'{len(nontrading)} transaction date(s) fall outside exchange sessions. Original dates retained; prior closes used.'))
    book, daily, flows = Book(), [], []
    income_totals = defaultdict(lambda: ZERO)
    latest, price_day = {}, {}
    symbols = list(instruments)
    start = transactions[0].day
    # Seed only prior quotes, never future ones.
    for symbol in symbols:
        earlier = [d for d in market.prices.get(symbol, {}) if d < start]
        if earlier:
            seed = max(earlier)
            latest[symbol] = market.prices[symbol][seed]
            price_day[symbol] = seed
    cum_buys = cum_sales = cum_income = ZERO
    previous_value, last_session = ZERO, max((d for d in sessions if d < start), default=None)
    day = start
    complete_previous = True
    final_values = {}
    while day <= end:
        for action in actions[day]:
            book.split(action.symbol, action.value)
        buys, sales, paid = ZERO, ZERO, ZERO
        for tx in by_day[day]:
            book.trade(tx)
            if tx.kind == 'BUY':
                value = tx.quantity * tx.price + tx.fees
                buys += value
                flows.append((day, -float(value)))
            elif tx.kind == 'SELL':
                value = tx.quantity * tx.price - tx.fees
                sales += value
                flows.append((day, float(value)))
        for item in payouts[day]:
            paid += item['amount']
            income_totals[item['symbol']] += item['amount']
            flows.append((day, float(item['amount'])))
        if day in sessions:
            last_session = day
        values, complete = {}, True
        for symbol in symbols:
            quote = market.prices.get(symbol, {}).get(day)
            if quote is not None and quote > 0 and quote.is_finite():
                latest[symbol], price_day[symbol] = quote, day
            if not book.quantity[symbol]:
                values[symbol] = ZERO
            elif symbol in latest and price_day[symbol] == last_session:
                values[symbol] = book.quantity[symbol] * latest[symbol]
            else:
                values[symbol] = None
                complete = False
        subtotal = sum((v for v in values.values() if v is not None), ZERO)
        value = subtotal if complete else None
        cum_buys += buys
        cum_sales += sales
        cum_income += paid
        denominator = previous_value + buys if previous_value is not None else ZERO
        daily_return = None
        if complete and complete_previous and denominator > 0:
            daily_return = float((value + sales + paid) / denominator - 1)
        daily.append({'date': day, 'value': float(value) if value is not None else None,
                      'priced_subtotal': float(subtotal), 'purchases': float(cum_buys),
                      'sale_proceeds': float(cum_sales), 'income': float(cum_income),
                      'profit': float(value + cum_sales + cum_income - cum_buys) if value is not None else None,
                      'daily_return': daily_return, 'complete': complete,
                      'buy_flow': float(buys), 'sale_flow': float(sales), 'income_flow': float(paid)})
        previous_value, complete_previous, final_values = value, complete, values
        day += timedelta(days=1)
    holdings = []
    for symbol, instrument in instruments.items():
        value, cost = final_values[symbol], book.cost[symbol]
        unrealized = value - cost if value is not None else None
        holdings.append({'symbol': symbol, 'investment': instrument.label, 'type': instrument.kind,
                         'sector': instrument.sector, 'quantity': float(book.quantity[symbol]),
                         'price': float(latest[symbol]) if symbol in latest else None,
                         'price_date': price_day.get(symbol), 'cost': float(cost),
                         'value': float(value) if value is not None else None,
                         'realized': float(book.realized[symbol]),
                         'unrealized': float(unrealized) if unrealized is not None else None,
                         'income': float(income_totals[symbol]),
                         'profit': float(book.realized[symbol] + unrealized + income_totals[symbol]) if unrealized is not None else None})
    if any(not d['complete'] for d in daily):
        issues.append(Issue('warning', 'Missing trading-day prices interrupt valuations and returns. Missing values are not zero.'))
    if estimates:
        issues.append(Issue('info', 'Estimated income uses eligible units and ex-dividend dates, not confirmed payment dates. XIRR timing is approximate.'))
    if not estimate:
        issues.append(Issue('warning', 'Online income estimation is off. Results exclude unrecorded income.'))
    return Analysis(transactions, instruments, holdings, daily, income, issues, conflicts, flows, market)

```

### `portfolio_compass/ai.py`

This is the implementation snapshot used by the application.

```python
"""Scoped conversational analysis. Groq selects evidence; it never supplies displayed facts.

The application renders approved templates only. All request words must belong to a
supported intent/portfolio alias, and unknown names are rejected before any API call.
"""

import json
import re
import threading
import time
from collections import deque

from .config import ai_available, setting
from .instruments import aliases, key
from .performance import period_metrics

REFUSAL = ('I can help with your uploaded portfolio, its performance, and the S&P 500 comparison '
           'in Portfolio Compass. Please ask a question within that scope.')
GLOSSARY = {
    'fifo': 'FIFO means first bought, first sold. A sale uses the cost of your oldest remaining purchase lots first.',
    'xirr': 'XIRR is an annualized return based on when you invested and received money. Estimated distributions use ex-dividend dates. Cash flows with multiple sign changes can make XIRR ambiguous.',
    'unrealized': 'Unrealized gain is the current value of investments still held minus their remaining purchase cost, including purchase fees.',
    'realized': 'Realized gain is sale proceeds after fees minus the FIFO cost of the units sold.',
    'income': 'Income combines recorded distributions with reconciled market-data estimates. Estimated income is not a confirmed payment and is not automatically reinvested.',
    'benchmark': 'The S&P 500 comparison uses SPY as an ETF proxy. Both return lines start at zero on the first selected close. Portfolio returns use the disclosed daily cash-flow approximation.',
    'upload': 'Upload one CSV with investment name or ticker, date, transaction type, quantity and unit price. Map its columns, confirm investment identities, review issues, and analyze. Recorded distributions use a cash-amount column.',
    'sell': 'This is decision support, not a trade recommendation. Review holdings with losses, incomplete valuation, or unusually large allocations before deciding whether any action is appropriate.',
}
WORDS = set(('a an the my our me i we you your please can could would show tell explain help understand '
             'what which how why is are was were did does do has have had with without and or of on in to for '
             'from than as at by about this that these those it its they their them all only much many '
             'portfolio compass app holding holdings investment investments stock stocks fund funds '
             'perform performed performing performance return returns annualized xirr fifo cost basis '
             'value worth current currently remaining own owned units shares quantity price prices '
             'profit profits loss losses gain gains realized unrealized sold sales sale proceeds purchases '
             'income dividend dividends distribution distributions estimated recorded total net fees '
             'largest biggest highest lowest smallest best worst contributed contribution allocation '
             'concentration concentrated percent percentage compare comparison versus vs benchmark market '
             'meaning mean means definition calculated calculate calculation explain summary summarize overview '
             'upload csv column columns map mapping import data dates date history historical today money '
             'more less most least different difference dollars invested understand doing shall sell selling').split())
ADVICE = re.compile(r'\b(should|recommend\w*|suggest\w*|buy|rebalance\w*|predict\w*|forecast\w*|tomorrow|target)\b', re.I)
_calls = deque()
_lock = threading.Lock()


def money(value):
    return 'Unavailable' if value is None else f'${value:,.2f}'


def route(question, instruments, last_symbol=None):
    if len(question) > 1000 or any(ord(c) < 32 for c in question):
        return None, REFUSAL
    text = key(question).replace('’', "'")
    if ADVICE.search(text):
        return None, 'Investment recommendations and predictions are unavailable. Ask about your recorded performance, holdings, or concentration instead.'
    found, ambiguous = set(), False
    alias_map = {}
    for symbol, item in instruments.items():
        for alias in aliases(item):
            alias_map.setdefault(alias, set()).add(symbol)
    benchmark = False
    for alias in ['s&p 500', 's&p500', 's and p 500', 'sp500', 'spy']:
        pattern = r'(?<!\w)' + re.escape(alias) + r'(?!\w)'
        if re.search(pattern, text):
            benchmark = True
            text = re.sub(pattern, ' benchmark ', text)
    for alias in sorted(alias_map, key=len, reverse=True):
        pattern = r'(?<!\w)' + re.escape(alias) + r'(?!\w)'
        if re.search(pattern, text):
            if len(alias_map[alias]) > 1:
                ambiguous = True
            found.update(alias_map[alias])
            text = re.sub(pattern, ' holding ', text)
    # Explicitly fail closed; irrelevant requests cannot ride along with a recognized ticker.
    if re.search(r'[^a-z\s?.!,\-\']', text):
        return None, REFUSAL
    text = text.replace("'s", '')
    tokens = set(re.findall(r'[a-z]+', text))
    if tokens - WORDS:
        return None, REFUSAL
    if ambiguous:
        return None, 'That name matches more than one investment in your portfolio. Please use its ticker.'
    if ('it' in tokens or 'its' in tokens) and not found:
        if last_symbol in instruments:
            found.add(last_symbol)
        else:
            return None, 'Please name the investment or use its ticker so I can answer from the correct holding.'
    topic = 'overview'
    if tokens & {'upload', 'csv', 'mapping', 'import', 'columns'}:
        topic = 'upload'
    elif tokens & {'fifo', 'xirr', 'unrealized', 'realized'}:
        topic = next(x for x in ['fifo', 'xirr', 'unrealized', 'realized'] if x in tokens)
    elif benchmark or tokens & {'benchmark', 'compare', 'comparison', 'versus'}:
        topic = 'benchmark'
    elif tokens & {'income', 'dividend', 'dividends', 'distribution', 'distributions'}:
        topic = 'income'
    elif tokens & {'allocation', 'concentration', 'concentrated', 'largest', 'biggest'}:
        topic = 'allocation'
    elif tokens & {'sell', 'selling'}:
        topic = 'sell'
    elif tokens & {'contributed', 'contribution', 'best', 'worst', 'highest', 'lowest'}:
        topic = 'contribution'
    elif tokens & {'profit', 'loss', 'gains', 'gain', 'performance', 'perform', 'return', 'returns'}:
        topic = 'performance'
    elif not tokens & {'summary', 'summarize', 'overview', 'holding', 'holdings', 'portfolio', 'value', 'worth', 'shares', 'quantity', 'own', 'cost', 'fees'}:
        return None, REFUSAL
    return {'topic': topic, 'symbols': sorted(found), 'definition': bool(tokens & {'meaning', 'mean', 'means', 'definition', 'explain'})}, ''


def evidence(analysis, intent):
    topic, selected = intent['topic'], intent['symbols']
    facts = []
    if topic in GLOSSARY:
        facts.append(GLOSSARY[topic])
    if topic == 'upload' or (intent['definition'] and topic in GLOSSARY):
        return {f'F{i}': text for i, text in enumerate(facts)}
    rows = [r for r in analysis.holdings if not selected or r['symbol'] in selected]
    end = analysis.daily[-1]
    if not selected:
        facts.append(f"As of {end['date']}, your investment value is {money(end['value'])}; total profit is {money(end['profit'])}. Brokerage cash is excluded.")
    if topic == 'xirr':
        metrics = period_metrics(analysis, analysis.daily[0]['date'], end['date'])
        rate = metrics['xirr']
        facts.append(f"Your annualized return is {rate:.2%}." if rate is not None else 'XIRR is unavailable: ' + metrics['reason'])
    elif topic == 'benchmark':
        from .performance import comparison
        series, reason = comparison(analysis, analysis.daily[0]['date'], end['date'])
        if series:
            last = series[-1]
            facts.append(f"Over {series[0]['date']} to {last['date']}, your rebased return is {last['Portfolio']:.2f}% and the S&P 500 proxy return is {last['S&P 500 (SPY proxy)']:.2f}%.")
        else:
            facts.append(reason)
    elif topic == 'allocation':
        active = sorted((r for r in rows if r['quantity'] > 0), key=lambda r: r['value'] or 0, reverse=True)
        for row in active[:5]:
            percent = row['value'] / end['value'] if row['value'] is not None and end['value'] else None
            facts.append(f"{row['investment']} represents {percent:.1%} of current investment value." if percent is not None else f"{row['investment']}: allocation is unavailable because valuation is incomplete.")
    elif topic == 'income':
        for row in rows[:10]:
            facts.append(f"{row['investment']}: recorded plus reconciled estimated income is {money(row['income'])}.")
    elif topic == 'sell':
        facts.append('I cannot decide what you should sell, but I can highlight holdings that deserve review. This is not a trade recommendation.')
        review_rows = sorted(
            rows,
            key=lambda r: (
                r['unrealized'] is not None and r['unrealized'] < 0,
                abs(r['unrealized']) if r['unrealized'] is not None else 0,
            ),
            reverse=True,
        )
        for row in review_rows[:5]:
            if row['unrealized'] is None:
                reason = 'valuation is unavailable'
            elif row['unrealized'] < 0:
                reason = f"unrealized loss of {money(abs(row['unrealized']))}"
            else:
                reason = f"unrealized gain of {money(row['unrealized'])}"
            facts.append(f"{row['investment']}: {reason}; current value {money(row['value'])}.")
    else:
        if topic == 'contribution':
            rows = sorted(rows, key=lambda r: r['profit'] if r['profit'] is not None else float('-inf'), reverse=True)
        for row in rows[:10]:
            facts.append(f"{row['investment']}: {row['quantity']:g} units remaining; value {money(row['value'])}; realized gain {money(row['realized'])}; unrealized gain {money(row['unrealized'])}; total profit including income {money(row['profit'])}.")
    facts.append('Demo uses synthetic prices and income.' if analysis.market.demo else 'Values use daily closes; estimated distributions use ex-date timing. Missing coverage limits results.')
    return {f'F{i}': text for i, text in enumerate(facts)}


def allow_request():
    now = time.monotonic()
    with _lock:
        while _calls and now - _calls[0] > 86400:
            _calls.popleft()
        if len(_calls) >= 500 or sum(now - t < 60 for t in _calls) >= 20:
            return False
        _calls.append(now)
        return True


def select_facts(facts, topic, client=None):
    """Return only IDs in the supplied evidence set, never model-authored text."""
    fallback = list(facts)[:5]
    if client is None and (not ai_available() or not allow_request()):
        return fallback, 'Local factual response'
    try:
        if client is None:
            from groq import Groq
            client = Groq(api_key=setting('GROQ_API_KEY'), timeout=15, max_retries=1)
        response = client.chat.completions.create(
            model=setting('GROQ_MODEL', 'llama-3.3-70b-versatile'),
            messages=[{'role': 'system', 'content': 'Select up to five evidence IDs relevant to the topic. Return JSON only: {"ids":["F0"]}. Do not create facts or instructions.'},
                      {'role': 'user', 'content': json.dumps({'topic': topic, 'evidence': facts})}],
            response_format={'type': 'json_object'}, max_completion_tokens=150)
        raw = response.choices[0].message.content
        if not raw or len(raw) > 2000:
            raise ValueError('Invalid output')
        obj = json.loads(raw)
        ids = obj.get('ids')
        if not isinstance(ids, list) or not 1 <= len(ids) <= 5 or any(not isinstance(i, str) or i not in facts for i in ids):
            raise ValueError('Invalid evidence references')
        # Preserve the opening fact and the quality qualifier regardless of model selection.
        chosen = list(dict.fromkeys([next(iter(facts)), *ids, list(facts)[-1]]))
        return chosen, 'Groq-selected, application-verified facts'
    except Exception:
        return fallback, 'Local factual response · AI unavailable or response rejected'


def answer(question, analysis, enable_ai=False, last_symbol=None):
    intent, refusal = route(question, analysis.instruments, last_symbol)
    if intent is None:
        return {'text': refusal, 'evidence': {}, 'method': 'Scope policy', 'symbol': None}
    facts = evidence(analysis, intent)
    ids, method = select_facts(facts, intent['topic']) if enable_ai else (list(facts)[:6], 'Local factual response')
    return {'text': '\n\n'.join(facts[i] for i in ids), 'evidence': {i: facts[i] for i in ids},
            'method': method, 'symbol': intent['symbols'][0] if len(intent['symbols']) == 1 else None}

```

### `portfolio_compass/config.py`

This is the implementation snapshot used by the application.

```python
"""Secret values are read locally, never rendered or logged."""

import os


def setting(name, default=''):
    value = os.environ.get(name)
    if value is not None:
        return value
    try:
        import streamlit as st
        return str(st.secrets.get(name, default))
    except (FileNotFoundError, KeyError):
        return default


def ai_available():
    return setting('COMPASS_AI_ENABLED', 'true').lower() == 'true' and bool(setting('GROQ_API_KEY'))

```

### `portfolio_compass/importing.py`

This is the implementation snapshot used by the application.

```python
"""Bounded CSV parsing. No silent row deletion, numeric coercion or date guessing."""

import csv
import io
import re
from datetime import date, datetime
from decimal import Decimal, InvalidOperation

from .models import MAX_BYTES, MAX_INSTRUMENTS, MAX_ROWS, Issue, Transaction, ZERO

ALIASES = {
    'instrument': ['code', 'stockcode', 'securitycode', 'ticker', 'symbol', 'stock', 'company', 'companyname', 'investment', 'security', 'fundname', 'name'],
    'name': ['companyname', 'securityname', 'fundname', 'name'],
    'date': ['date', 'tradedate', 'transactiondate', 'postingdate'],
    'type': ['type', 'transactiontype', 'action', 'side', 'activity'],
    'quantity': ['quantity', 'qty', 'shares', 'units'],
    'price': ['price', 'unitprice', 'shareprice', 'tradeprice'],
    'amount': ['cashamount', 'amount', 'dividend', 'distributionamount'],
    'fees': ['fees', 'fee', 'commission', 'commissions'],
    'id': ['id', 'transactionid', 'tradeid'],
    'sequence': ['sequence', 'order', 'sequencenumber'],
}
TYPES = {'BUY': 'BUY', 'BOUGHT': 'BUY', 'PURCHASE': 'BUY', 'B': 'BUY',
         'SELL': 'SELL', 'SOLD': 'SELL', 'SALE': 'SELL', 'S': 'SELL',
         'DIVIDEND': 'DISTRIBUTION', 'DIV': 'DISTRIBUTION', 'DISTRIBUTION': 'DISTRIBUTION',
         'CASH DIVIDEND': 'DISTRIBUTION'}


def read_csv(data: bytes, encoding='utf-8-sig', delimiter='Auto'):
    if not data or len(data) > MAX_BYTES:
        raise ValueError('Upload a nonempty CSV no larger than 10 MB.')
    try:
        text = data.decode(encoding)
    except UnicodeDecodeError as exc:
        raise ValueError('This encoding could not read the file. Choose its encoding explicitly.') from exc
    if '\x00' in text:
        raise ValueError('Binary content is not a supported CSV.')
    if delimiter == 'Auto':
        try:
            delimiter = csv.Sniffer().sniff(text[:8192], delimiters=',;\t|').delimiter
        except csv.Error:
            delimiter = ','
    reader = csv.reader(io.StringIO(text), delimiter=delimiter, strict=True)
    try:
        headers = next(reader)
        headers = [h.strip() for h in headers]
        if len(headers) > 40 or len(headers) < 2 or any(not h or len(h) > 100 for h in headers):
            raise ValueError('Use 2–40 nonempty column names, each at most 100 characters.')
        if len(set(headers)) != len(headers):
            raise ValueError('Column names must be unique.')
        rows = []
        for row in reader:
            if not row or all(not cell.strip() for cell in row):
                continue
            if len(row) != len(headers) or any(len(cell) > 500 for cell in row):
                raise ValueError(f'CSV record ending on line {reader.line_num} has invalid columns or oversized cells.')
            rows.append({**dict(zip(headers, (cell.strip() for cell in row))), '_row': reader.line_num})
            if len(rows) > MAX_ROWS:
                raise ValueError('The limit is 10,000 transactions.')
    except (csv.Error, StopIteration) as exc:
        raise ValueError('The CSV structure could not be read.') from exc
    if not rows:
        raise ValueError('The CSV contains no transaction rows.')
    return headers, rows


def suggest_mapping(headers):
    clean = {re.sub(r'[^a-z0-9]', '', h.lower()): h for h in headers}
    result = {field: next((clean[a] for a in aliases if a in clean), '') for field, aliases in ALIASES.items()}
    if result['name'] == result['instrument']:
        result['name'] = ''
    return result


def number(value, convention='US', default=None):
    if value == '' and default is not None:
        return default
    text = str(value).strip().removeprefix('$').strip()
    pattern = r'\+?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?' if convention == 'US' else r'\+?(?:\d{1,3}(?:\.\d{3})+|\d+)(?:,\d+)?'
    if not re.fullmatch(pattern, text):
        raise ValueError('Use positive numeric values in the selected number format; encode direction in transaction type.')
    text = text.replace(',', '') if convention == 'US' else text.replace('.', '').replace(',', '.')
    try:
        value = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError('Invalid numeric value.') from exc
    if not value.is_finite() or value > Decimal('1000000000000') or value.as_tuple().exponent < -10:
        raise ValueError('Numeric value exceeds supported bounds (1 trillion; 10 decimal places).')
    return value


def parse_day(text, date_format):
    formats = {'ISO (YYYY-MM-DD)': '%Y-%m-%d', 'US (MM/DD/YYYY)': '%m/%d/%Y', 'Day first (DD/MM/YYYY)': '%d/%m/%Y'}
    try:
        day = datetime.strptime(text, formats[date_format]).date()
    except (ValueError, KeyError) as exc:
        raise ValueError('Date does not match the selected format.') from exc
    if day > date.today() or day < date(1990, 1, 1):
        raise ValueError('Dates must be between 1990-01-01 and today.')
    return day


def normalize(rows, mapping, resolved, date_format='ISO (YYYY-MM-DD)', convention='US', keep_duplicates=False):
    issues, result = [], []
    if any(not mapping.get(k) for k in ['instrument', 'date', 'type']):
        return [], [Issue('error', 'Map the investment, date, and transaction type columns.')]
    columns = [v for v in mapping.values() if v]
    if len(columns) != len(set(columns)):
        return [], [Issue('error', 'Each input column can map to only one field.')]
    seen_ids, seen_rows = set(), set()
    for row in rows:
        def get(key):
            return row.get(mapping.get(key, ''), '')
        try:
            original = get('instrument')
            instrument = resolved.get(original)
            if instrument is None:
                raise ValueError('Confirm the investment identity before analysis.')
            if mapping.get('name') and get('name'):
                other = resolved.get(get('name'))
                if other is None or other.symbol != instrument.symbol:
                    raise ValueError('Ticker and company/fund name do not resolve to the same security.')
            kind = TYPES.get(get('type').upper())
            if not kind:
                raise ValueError('Unsupported transaction type. Use BUY, SELL or DISTRIBUTION.')
            day = parse_day(get('date'), date_format)
            trade = kind != 'DISTRIBUTION'
            qty = number(get('quantity'), convention) if trade else ZERO
            price = number(get('price'), convention) if trade else ZERO
            amount = number(get('amount'), convention) if not trade else ZERO
            fees = number(get('fees'), convention, ZERO)
            if (trade and (qty <= 0 or price <= 0)) or (not trade and amount <= 0):
                raise ValueError('Trade quantity/price and distribution amount must be greater than zero.')
            if trade and kind == 'SELL' and fees > qty * price:
                raise ValueError('Sale fees cannot exceed proceeds.')
            if not trade and fees:
                raise ValueError('For distributions supply the net cash amount with zero separate fees.')
            identifier = get('id') or f'row-{row["_row"]}'
            if identifier in seen_ids:
                raise ValueError('Duplicate transaction ID; correct it rather than silently dropping a row.')
            seen_ids.add(identifier)
            signature = (instrument.symbol, day, kind, qty, price, fees, amount)
            if signature in seen_rows:
                issues.append(Issue('warning' if keep_duplicates else 'error', 'Possible duplicate trade/distribution. Confirm that identical rows are intentional.', row['_row']))
            seen_rows.add(signature)
            result.append(Transaction(identifier, day, instrument.symbol, kind, qty, price, fees, amount, row['_row'], original, number(get('sequence'), convention, ZERO)))
        except ValueError as exc:
            issues.append(Issue('error', str(exc), row['_row']))
    if len({t.symbol for t in result}) > MAX_INSTRUMENTS:
        issues.append(Issue('error', 'The limit is 100 resolved portfolio investments.'))
    if not mapping.get('fees'):
        issues.append(Issue('warning', 'No fee column: fees are assumed to be zero.'))
    if not mapping.get('sequence') and len({(t.day, t.symbol) for t in result}) < len(result):
        issues.append(Issue('warning', 'Same-day events use original file order. Map a sequence column if needed.'))
    return sorted(result, key=lambda t: (t.day, t.sequence, t.row)), issues


def export_csv(records):
    if not records:
        return b''
    output = io.StringIO()
    writer = csv.DictWriter(output, fieldnames=list(records[0]))
    writer.writeheader()
    for record in records:
        safe = {}
        for key, value in record.items():
            if isinstance(value, str) and value.lstrip().startswith(('=', '+', '-', '@', '\t', '\r')):
                value = "'" + value
            safe[key] = value
        writer.writerow(safe)
    return output.getvalue().encode('utf-8-sig')

```

### `portfolio_compass/instruments.py`

This is the implementation snapshot used by the application.

```python
"""Security identities and verified aliases; fuzzy matches are suggestions only."""

import re
import unicodedata
from difflib import SequenceMatcher

from .models import Instrument

CATALOG = (
    Instrument('AAPL', 'Apple', aliases=('Apple Inc.', 'Apple Inc'), sector='Technology'),
    Instrument('MSFT', 'Microsoft', aliases=('Microsoft Corporation', 'Microsoft Corp'), sector='Technology'),
    Instrument('TSLA', 'Tesla', aliases=('Tesla Inc.', 'Tesla Inc'), sector='Consumer cyclical'),
    Instrument('SPY', 'SPDR S&P 500 ETF Trust', 'ETF', 'NYSE Arca', aliases=('SPY',)),
)


def key(text):
    return ' '.join(unicodedata.normalize('NFKC', str(text)).casefold().split())


def aliases(instrument):
    return {key(x) for x in (instrument.symbol, instrument.name, *instrument.aliases)}


def valid_query(text):
    return bool(re.fullmatch(r"[\w .&'()/-]{1,100}", text, re.UNICODE)) and not any(ord(c) < 32 for c in text)


def catalog_candidates(text):
    if not valid_query(text):
        return []
    query = key(text)
    exact = [item for item in CATALOG if query in aliases(item)]
    if exact:
        return exact
    return [item for item in CATALOG if any(query in a or SequenceMatcher(None, query, a).ratio() > .72 for a in aliases(item))]


def verified_candidate(info, symbol):
    if not re.fullmatch(r'[A-Z0-9^][A-Z0-9.\-^=]{0,19}', symbol):
        return None
    kinds = {'EQUITY': 'Stock', 'ETF': 'ETF', 'MUTUALFUND': 'Fund'}
    kind = kinds.get(info.get('quoteType'))
    exchange = info.get('exchange', '')
    if not kind or info.get('currency') != 'USD' or exchange not in {'NMS', 'NGM', 'NCM', 'NYQ', 'ASE', 'PCX', 'BTS', 'BATS', 'NYS', 'NAS', 'PNK', 'OEM', 'OBB', 'OQB', 'OQX', 'NMF'}:
        return None
    name = info.get('longName') or info.get('shortName')
    if not name or len(name) > 200 or any(ord(c) < 32 for c in name):
        return None
    known = next((x for x in CATALOG if x.symbol == symbol), None)
    extra = tuple({str(info.get('shortName') or name), *(known.aliases if known else ()), *( (known.name,) if known else ())})
    return Instrument(symbol, name, kind, info.get('fullExchangeName', exchange), 'USD', extra, info.get('sector', 'Unknown'), 'Yahoo metadata')


def resolve_batch(inputs, provider_lookup):
    """One batch: auto-resolve codes/clear names; return only uncertain names for review."""
    resolved, ambiguous, errors = {}, {}, {}
    for raw in dict.fromkeys(inputs):
        local = catalog_candidates(raw)
        exact = [item for item in local if key(raw) in aliases(item)]
        if len(exact) == 1:
            resolved[raw] = exact[0]
            continue
        candidates, error = provider_lookup(raw)
        candidates = list({item.symbol: item for item in candidates}.values())
        codes = [item for item in candidates if key(raw).replace('.', '-') == key(item.symbol).replace('.', '-')]
        if len(codes) == 1:
            resolved[raw] = codes[0]
        elif re.fullmatch(r'[A-Z]{1,6}(?:[.-][A-Z])?', raw):
            errors[raw] = error or 'Code could not be verified. Correct the code or retry validation.'
        elif len(candidates) == 1:
            resolved[raw] = candidates[0]
        elif candidates:
            ambiguous[raw] = candidates
        else:
            errors[raw] = error or 'No supported investment matched this name. Correct it and validate again.'
    return resolved, ambiguous, errors

```

### `portfolio_compass/market_data.py`

This is the implementation snapshot used by the application.

```python
"""Bounded public-data adapters. Private transactions are never cached here."""

import math
import re
import time
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from functools import lru_cache
from pathlib import Path

import exchange_calendars as xcals
import pandas as pd

from .instruments import valid_query, verified_candidate
from .models import Action, Issue, MarketData


@lru_cache(maxsize=8)
def calendar(start, end):
    return xcals.get_calendar('XNYS', start=str(start), end=str(end))


def session_dates(start, end):
    return set(calendar(start - timedelta(days=10), end + timedelta(days=10)).sessions.date)


def last_completed_session():
    now = datetime.now(timezone.utc)
    cal = calendar(now.date() - timedelta(days=15), now.date() + timedelta(days=5))
    return max(s.date() for s in cal.sessions if cal.session_close(s).to_pydatetime() + timedelta(minutes=20) <= now)


def _yf():
    import yfinance as yf
    # This contains public quote-provider cookies/timezone metadata, not portfolios.
    cache = Path(__file__).resolve().parents[1] / 'artifacts' / 'market-cache'
    cache.mkdir(parents=True, exist_ok=True)
    yf.set_tz_cache_location(str(cache))
    return yf


def lookup(query):
    if not valid_query(query):
        return [], 'Use a short company/fund name or ticker without control characters.'
    return _lookup(query, int(time.time() // 3600))


@lru_cache(maxsize=400)
def _lookup(query, bucket):
    try:
        yf = _yf()
        if re.fullmatch(r'[A-Za-z]{1,6}(?:[.-][A-Za-z])?', query):
            symbol = query.upper().replace('.', '-')
            try:
                info = yf.Ticker(symbol).get_info()
                item = verified_candidate(info, symbol)
                if item and str(info.get('symbol', symbol)).upper() == symbol:
                    return [item], ''
            except Exception:
                pass  # A short common name may need the name search below.
        search = yf.Search(query, max_results=6, news_count=0, enable_fuzzy_query=False, timeout=12)
        symbols = list(dict.fromkeys(q.get('symbol', '') for q in search.quotes))[:6]
        # Search results alone do not authorize an instrument.
        result = []
        for symbol in symbols:
            if not valid_query(symbol):
                continue
            info = yf.Ticker(symbol).get_info()
            item = verified_candidate(info, symbol)
            if item:
                result.append(item)
        return result, '' if result else 'No supported US/USD security matched. Try an exact ticker.'
    except Exception:
        return [], 'Security lookup is unavailable. Retry later; no identity was guessed.'


@lru_cache(maxsize=400)
def _history(symbol, start, end, bucket):
    return _yf().Ticker(symbol).history(start=start, end=end, auto_adjust=False,
                                       actions=True, repair=False, timeout=15, raise_errors=True)


def normalize_history(symbol, frame):
    """Yahoo closes/dividends are split-adjusted. Undo future splits for event-date units."""
    if frame.empty or 'Close' not in frame:
        raise ValueError('No price history.')
    prices, actions, benchmark = {}, [], {}
    future_factor = Decimal('1')
    for timestamp, row in frame.sort_index(ascending=False).iterrows():
        day = timestamp.date()
        close = row.get('Close')
        if pd.notna(close) and math.isfinite(float(close)) and close > 0:
            prices[day] = Decimal(str(close)) * future_factor
        adjusted = row.get('Adj Close')
        if pd.notna(adjusted) and math.isfinite(float(adjusted)) and adjusted > 0:
            benchmark[day] = Decimal(str(adjusted))
        dividend = row.get('Dividends', 0)
        if pd.notna(dividend) and dividend > 0:
            actions.append(Action(f'{symbol}:{day}:div', day, symbol, 'DIVIDEND', Decimal(str(dividend)) * future_factor))
        ratio = row.get('Stock Splits', 0)
        if pd.notna(ratio) and ratio > 0:
            actions.append(Action(f'{symbol}:{day}:split', day, symbol, 'SPLIT', Decimal(str(ratio))))
            future_factor *= Decimal(str(ratio))
    return prices, sorted(actions, key=lambda x: x.day), benchmark


def fill_calendar_gaps(prices, start, end, sessions):
    """Fill weekends/holidays only. A missing exchange session remains missing."""
    output, previous = {}, None
    for day in pd.date_range(start, end).date:
        if day in prices:
            previous = prices[day]
        elif day in sessions:
            previous = None
        if previous is not None:
            output[day] = previous
    return output


def fetch_market(symbols, start, end, refresh=False):
    if len(symbols) > 100:
        raise ValueError('Maximum 100 portfolio securities.')
    result = MarketData(as_of=end)
    bucket = int(time.time() // 900)
    if refresh:
        _history.cache_clear()
    fetch_start = start - timedelta(days=10)
    sessions = session_dates(fetch_start, end)
    for symbol in dict.fromkeys([*symbols, 'SPY']):
        try:
            frame = _history(symbol, str(fetch_start), str(end + timedelta(days=1)), bucket)
            prices, actions, adjusted = normalize_history(symbol, frame)
            if symbol in symbols:
                result.prices[symbol] = prices
                result.actions.extend(actions)
                if not {'Dividends', 'Stock Splits'}.issubset(frame.columns):
                    result.income_complete = False
                    result.issues.append(Issue('warning', f'{symbol}: corporate-action coverage is unknown.'))
                if 'Capital Gains' in frame and frame['Capital Gains'].fillna(0).ne(0).any():
                    result.income_complete = False
                    result.issues.append(Issue('warning', f'{symbol}: capital-gain distributions require manual records; total-return comparison disabled.'))
            if symbol == 'SPY':
                result.benchmark = fill_calendar_gaps(adjusted, fetch_start, end, sessions)
        except Exception:
            result.issues.append(Issue('warning', f'{symbol}: market history is unavailable. Retry later.'))
            if symbol in symbols:
                result.income_complete = False
    return result


def demo_market(start=date(2024, 1, 3), end=date(2026, 10, 9)):
    """Reproducible synthetic fixture, deliberately not historical market claims."""
    result = MarketData(demo=True, as_of=end)
    sessions = sorted(d for d in session_dates(start - timedelta(days=10), end) if start - timedelta(days=10) <= d <= end)
    parameters = {'AAPL': (180, .00032, 7), 'MSFT': (365, .00028, 16), 'TSLA': (245, -.0001, 39), 'SPY': (470, .0003, 10)}
    for symbol, (base, slope, wave) in parameters.items():
        prices = {day: Decimal(str(round(base * math.exp(slope * n) + wave * math.sin(n / 35), 4))) for n, day in enumerate(sessions)}
        if symbol != 'SPY':
            result.prices[symbol] = prices
        else:
            result.benchmark = fill_calendar_gaps(prices, sessions[0], end, set(sessions))
    for year in (2024, 2025, 2026):
        for month in (2, 5, 8, 11):
            proposed = date(year, month, 15)
            dates = [d for d in sessions if d >= proposed]
            if not dates:
                continue
            for symbol, amount in [('AAPL', '.25'), ('MSFT', '.80')]:
                day = dates[0]
                result.actions.append(Action(f'{symbol}:{day}:div', day, symbol, 'DIVIDEND', Decimal(amount)))
    result.issues.append(Issue('info', 'Synthetic offline prices and distributions through October 9, 2026. Not actual market performance.'))
    return result

```

### `portfolio_compass/models.py`

This is the implementation snapshot used by the application.

```python
"""Value objects shared by the UI, accounting and provider boundaries."""

from dataclasses import dataclass, field
from datetime import date
from decimal import Decimal

ZERO = Decimal('0')
MAX_ROWS = 10_000
MAX_INSTRUMENTS = 100
MAX_BYTES = 10 * 1024 * 1024


@dataclass(frozen=True)
class Instrument:
    symbol: str
    name: str
    kind: str = 'Stock'
    exchange: str = 'NASDAQ'
    currency: str = 'USD'
    aliases: tuple[str, ...] = ()
    sector: str = 'Unknown'
    source: str = 'Verified catalog'

    @property
    def label(self):
        return f'{self.name} ({self.symbol})'


@dataclass(frozen=True)
class Transaction:
    id: str
    day: date
    symbol: str
    kind: str
    quantity: Decimal = ZERO
    price: Decimal = ZERO
    fees: Decimal = ZERO
    amount: Decimal = ZERO
    row: int = 0
    original: str = ''
    sequence: Decimal = ZERO


@dataclass(frozen=True)
class Issue:
    severity: str
    message: str
    row: int = 0


@dataclass(frozen=True)
class Action:
    id: str
    day: date
    symbol: str
    kind: str
    value: Decimal


@dataclass
class MarketData:
    prices: dict[str, dict[date, Decimal]] = field(default_factory=dict)
    actions: list[Action] = field(default_factory=list)
    benchmark: dict[date, Decimal] = field(default_factory=dict)
    issues: list[Issue] = field(default_factory=list)
    income_complete: bool = True
    demo: bool = False
    as_of: date | None = None


@dataclass
class Analysis:
    transactions: list[Transaction]
    instruments: dict[str, Instrument]
    holdings: list[dict]
    daily: list[dict]
    income: list[dict]
    issues: list[Issue]
    conflicts: list[dict]
    cashflows: list[tuple[date, float]]
    market: MarketData

    @property
    def complete(self):
        return bool(self.daily) and self.daily[-1]['complete'] and not self.conflicts

```

### `portfolio_compass/performance.py`

This is the implementation snapshot used by the application.

```python
"""Investor return solving and selected-period comparisons."""

from collections import defaultdict
from datetime import timedelta

import numpy as np
from scipy.optimize import brentq


def xirr(cashflows):
    """Find numerically isolated ACT/365 roots over log(1+r) in [-20, 20].

    Multiple cash-flow sign changes only imply possible multiple roots. Scan
    both NPV and its derivative to split intervals at extrema, including
    touching roots and close pairs that a sign-only grid could overlook.
    This is a bounded numerical search, not a proof of global uniqueness.
    """
    merged = defaultdict(float)
    for day, amount in cashflows:
        if not np.isfinite(amount):
            return None, 'Cash flows must contain finite amounts.'
        merged[day] += amount
    ordered = sorted((d, a) for d, a in merged.items() if abs(a) > 1e-10)
    if len(ordered) < 2 or not any(a < 0 for _, a in ordered) or not any(a > 0 for _, a in ordered):
        return None, 'Needs dated investments and proceeds or a positive ending value.'
    amounts = np.array([a for _, a in ordered], dtype=float)
    years = np.array([(d - ordered[0][0]).days / 365 for d, _ in ordered])
    scale = float(np.max(np.abs(amounts)))
    amounts /= scale

    def evaluate(log_rate, derivative=False):
        exponent = -log_rate * years
        weights = np.exp(exponent - max(float(exponent.max()), 0))
        coefficients = -years * amounts if derivative else amounts
        denominator = float(np.dot(np.abs(coefficients), weights))
        # Relative residual avoids accepting underflowed raw NPV as a root.
        return float(np.dot(coefficients, weights)) / denominator if denominator else np.nan

    def npv(log_rate):
        return evaluate(log_rate)

    try:
        grid = np.linspace(-20, 20, 4097)
        derivative = np.array([evaluate(x, True) for x in grid])
        extrema = []
        for index in range(len(grid) - 1):
            left, right = derivative[index:index + 2]
            if left == 0:
                extrema.append(float(grid[index]))
            elif np.isfinite(left) and np.isfinite(right) and np.signbit(left) != np.signbit(right):
                extrema.append(brentq(lambda x: evaluate(x, True), grid[index], grid[index + 1], xtol=1e-13))
        points = sorted(set([*grid, *extrema]))
        roots = []

        def add_root(root):
            if abs(npv(root)) < 1e-10 and not any(abs(root - old) < 1e-8 for old in roots):
                roots.append(root)

        for point in extrema:
            if abs(npv(point)) < 1e-12:
                add_root(point)
        values = [npv(x) for x in points]
        for index, (left, right) in enumerate(zip(values, values[1:])):
            if left == 0:
                add_root(points[index])
            elif np.isfinite(left) and np.isfinite(right) and np.signbit(left) != np.signbit(right):
                add_root(brentq(npv, points[index], points[index + 1], xtol=1e-13))
        if values[-1] == 0:
            add_root(points[-1])
        if len(roots) > 1:
            return None, 'Multiple annualized returns were detected; XIRR is ambiguous for these cash flows.'
        if not roots:
            return None, 'No reliable annualized return was found in the supported numerical range.'
        return float(np.expm1(roots[0])), ('Annualized, ACT/365. One root detected in the bounded numerical search; '
                                         'estimated distributions use ex-date timing.')
    except (ValueError, OverflowError):
        return None, 'No reliable annualized return in the supported numerical range.'


def period_metrics(analysis, start, end):
    rows = [d for d in analysis.daily if start <= d['date'] <= end]
    if not rows:
        return {'xirr': None, 'reason': 'No dates in range.', 'profit': None, 'rows': []}
    before = next((d for d in analysis.daily if d['date'] == start - timedelta(days=1)), None)
    opening = before['value'] if before else 0.0
    ending = rows[-1]['value']
    if opening is None or ending is None or analysis.conflicts:
        return {'xirr': None, 'reason': 'Opening/closing valuation or income reconciliation is incomplete.', 'profit': None, 'rows': rows}
    flows = [(d, amount) for d, amount in analysis.cashflows if start <= d <= end]
    if opening:
        flows.insert(0, (start - timedelta(days=1), -opening))
    flows.append((end, ending))
    rate, reason = xirr(flows)
    profit = ending - opening + sum(amount for d, amount in analysis.cashflows if start <= d <= end)
    return {'xirr': rate, 'reason': reason, 'profit': profit, 'rows': rows, 'opening': opening,
            'purchases': sum(r['buy_flow'] for r in rows), 'sales': sum(r['sale_flow'] for r in rows),
            'income': sum(r['income_flow'] for r in rows)}


def comparison(analysis, start, end):
    rows = [d for d in analysis.daily if start <= d['date'] <= end]
    if not rows or analysis.conflicts or not analysis.market.income_complete:
        return [], 'Comparison requires complete income coverage and reconciled distributions.'
    if any(r['daily_return'] is None for r in rows):
        return [], 'Choose a continuous invested period with complete prices; gaps and zero capital break the series.'
    # Both series begin at zero on the first selected date; growth after that date is compared.
    dates = [r['date'] for r in rows]
    available = analysis.market.benchmark
    values, previous = {}, None
    for day in sorted(set(available) | set(dates)):
        if day in available:
            previous = available[day]
        if day in dates:
            values[day] = previous
    if not values or any(values[d] is None for d in dates):
        return [], 'S&P 500 proxy prices are unavailable for this period.'
    # Provider adapter supplies only calendar-gap fill; missing sessions are not backfilled here.
    if any(d not in available for d in dates):
        return [], 'S&P 500 proxy coverage contains a gap.'
    base = float(values[dates[0]])
    growth, result = 1.0, []
    for index, row in enumerate(rows):
        if index:
            growth *= 1 + row['daily_return']
        result.append({'date': row['date'], 'Portfolio': (growth - 1) * 100,
                       'S&P 500 (SPY proxy)': (float(values[row['date']]) / base - 1) * 100})
    return result, 'Both lines start at 0% at the first selected close; subsequent daily returns are compounded.'

```

### `portfolio_compass/sample_data_generator.py`

This is the implementation snapshot used by the application.

```python
"""Generate synthetic investment transactions for Portfolio Compass.

The instrument universe is discovered from Yahoo Finance through yfinance.
Generated prices and transactions are synthetic fixtures; only the selected
instrument identities and current reference prices come from the network.
"""

from __future__ import annotations

import argparse
import csv
import random
import re
import tkinter as tk
from dataclasses import dataclass
from datetime import date, timedelta
from decimal import Decimal
from pathlib import Path
from tkinter import filedialog
from typing import Literal, Sequence

InputShape = Literal["canonical", "broker", "name-based", "mixed"]
SHAPES: tuple[InputShape, ...] = ("canonical", "broker", "name-based", "mixed")


@dataclass(frozen=True)
class MarketInstrument:
    symbol: str
    name: str
    quote_type: str
    reference_price: Decimal


# Used only when Yahoo is unavailable. These are deliberately broader than
# the original three-instrument demo and are never read from a sample file.
OFFLINE_FALLBACK = (
    MarketInstrument("SPY", "SPDR S&P 500 ETF Trust", "ETF", Decimal("500")),
    MarketInstrument("QQQ", "Invesco QQQ Trust", "ETF", Decimal("450")),
    MarketInstrument("VTI", "Vanguard Total Stock Market ETF", "ETF", Decimal("270")),
    MarketInstrument("NVDA", "NVIDIA", "EQUITY", Decimal("120")),
    MarketInstrument("AMD", "Advanced Micro Devices", "EQUITY", Decimal("160")),
    MarketInstrument("GOOGL", "Alphabet", "EQUITY", Decimal("180")),
    MarketInstrument("AMZN", "Amazon.com", "EQUITY", Decimal("220")),
    MarketInstrument("JPM", "JPMorgan Chase", "EQUITY", Decimal("300")),
    MarketInstrument("COST", "Costco Wholesale", "EQUITY", Decimal("950")),
    MarketInstrument("KO", "Coca-Cola", "EQUITY", Decimal("70")),
)


@dataclass(frozen=True)
class GeneratorConfig:
    """Options for a deterministic transaction fixture."""

    count: int = 24
    instrument_count: int = 3
    shape: InputShape = "canonical"
    seed: int | None = 42
    include_distributions: bool = True

    def validate(self) -> None:
        if not 3 <= self.count <= 10_000:
            raise ValueError("count must be between 3 and 10,000 transactions")
        if not 1 <= self.instrument_count <= 25:
            raise ValueError("instrument_count must be between 1 and 25")
        if self.instrument_count > self.count:
            raise ValueError("instrument_count cannot exceed the total transaction count")
        if self.shape not in SHAPES:
            raise ValueError(f"shape must be one of: {', '.join(SHAPES)}")


def _sample_instruments(candidates: Sequence[MarketInstrument], count: int, seed: int | None) -> list[MarketInstrument]:
    """Select instruments, reserving one ETF when a portfolio has 2+ holdings."""

    unique = {item.symbol: item for item in candidates}
    if len(unique) < count:
        raise RuntimeError(f"Only {len(unique)} usable instruments are available; {count} requested")
    rng = random.Random(seed)
    values = list(unique.values())
    if count == 1:
        return [rng.choice(values)]
    etfs = [item for item in values if item.quote_type == "ETF"]
    if not etfs:
        raise RuntimeError("The discovered universe contains no ETF; try again")
    selected = [rng.choice(etfs)]
    remaining = [item for item in values if item.symbol != selected[0].symbol]
    selected.extend(rng.sample(remaining, count - 1))
    return selected


def discover_instruments(count: int = 25, seed: int | None = None) -> list[MarketInstrument]:
    """Discover random US stocks/ETFs from Yahoo Finance screener results.

    yfinance's ``screen`` endpoint returns live screener quotes. We combine
    several predefined screens, remove duplicate symbols, filter to USD
    equity/ETF quotes with a usable price, then sample the requested count.
    """

    if not 1 <= count <= 100:
        raise ValueError("count must be between 1 and 100")
    try:
        import yfinance as yf

        candidates: dict[str, MarketInstrument] = {}
        for screen_name in ("most_actives", "day_gainers", "portfolio_anchors", "high_yield_bond"):
            response = yf.screen(screen_name, count=250)
            for quote in response.get("quotes", []):
                symbol = str(quote.get("symbol", "")).upper()
                quote_type = str(quote.get("quoteType", "")).upper()
                currency = str(quote.get("currency", "USD")).upper()
                raw_price = quote.get("regularMarketPrice") or quote.get("regularMarketPreviousClose")
                name = str(quote.get("longName") or quote.get("shortName") or "").strip()
                if (not re.fullmatch(r"[A-Z0-9][A-Z0-9.\-^=]{0,19}", symbol)
                        or quote_type not in {"EQUITY", "ETF"} or currency != "USD" or not name):
                    continue
                try:
                    price = Decimal(str(raw_price))
                except (ValueError, TypeError):
                    continue
                if price.is_finite() and price > 0:
                    candidates[symbol] = MarketInstrument(symbol, name, quote_type, price)
        if len(candidates) < count:
            raise RuntimeError(f"Yahoo Finance returned only {len(candidates)} usable instruments")
        return _sample_instruments(list(candidates.values()), count, seed)
    except Exception as exc:
        raise RuntimeError(f"Unable to discover instruments from Yahoo Finance: {exc}") from exc


def _business_days(start: date, end: date) -> list[date]:
    return [start + timedelta(days=offset) for offset in range((end - start).days + 1)
            if (start + timedelta(days=offset)).weekday() < 5]


def _raw_row(sequence: int, instrument: MarketInstrument, day: date, kind: str, quantity: int,
             price: Decimal | int, fees: Decimal | int, amount: Decimal | int) -> dict[str, str]:
    return {
        "ticker": instrument.symbol,
        "company": instrument.name,
        "date": day.isoformat(),
        "transaction_type": kind,
        "quantity": str(quantity),
        "price": f"{Decimal(str(price)):.2f}" if price else "",
        "fees": f"{Decimal(str(fees)):.2f}" if fees else "",
        "amount": f"{Decimal(str(amount)):.2f}" if amount else "",
        "transaction_id": f"TX-{sequence:05d}",
        "sequence": str(sequence),
    }


def _shape_row(row: dict[str, str], shape: InputShape) -> dict[str, str]:
    if shape == "canonical":
        return {"ticker": row["ticker"], "date": row["date"], "transaction_type": row["transaction_type"],
                "quantity": row["quantity"], "price": row["price"], "fees": row["fees"], "amount": row["amount"],
                "transaction_id": row["transaction_id"], "sequence": row["sequence"]}
    if shape == "broker":
        return {"symbol": row["ticker"], "trade_date": row["date"], "action": row["transaction_type"],
                "shares": row["quantity"], "unit_price": row["price"], "commission": row["fees"],
                "cash_amount": row["amount"], "trade_id": row["transaction_id"], "order": row["sequence"]}
    if shape == "name-based":
        return {"company_name": row["company"], "transaction_date": row["date"], "type": row["transaction_type"],
                "units": row["quantity"], "trade_price": row["price"], "fees": row["fees"],
                "amount": row["amount"], "id": row["transaction_id"], "order": row["sequence"]}
    return {"symbol": row["ticker"], "company_name": row["company"], "date": row["date"],
            "activity": row["transaction_type"], "shares": row["quantity"], "price": row["price"],
            "fees": row["fees"], "cash_amount": row["amount"], "id": row["transaction_id"],
            "sequence_number": row["sequence"]}


def _transactions(config: GeneratorConfig, instruments: Sequence[MarketInstrument]) -> list[dict[str, str]]:
    rng = random.Random(config.seed)
    days = _business_days(date.today() - timedelta(days=900), date.today() - timedelta(days=1))
    selected = list(instruments)
    rng.shuffle(selected)
    held = {item.symbol: 0 for item in selected}
    rows: list[dict[str, str]] = []
    sequence = 1

    for instrument in selected:
        quantity = rng.choice((8, 10, 12, 15))
        price = instrument.reference_price * Decimal(str(rng.uniform(0.90, 1.10))).quantize(Decimal("0.01"))
        rows.append(_raw_row(sequence, instrument, days.pop(0), "BUY", quantity, price, 0, 0))
        held[instrument.symbol] = quantity
        sequence += 1

    while len(rows) < config.count:
        instrument = rng.choice(selected)
        day = days.pop(rng.randrange(len(days)))
        price = instrument.reference_price * Decimal(str(rng.uniform(0.88, 1.16))).quantize(Decimal("0.01"))
        roll = rng.random()
        if config.include_distributions and roll < 0.16:
            amount = Decimal(str(rng.choice(("0.22", "0.25", "0.80")))) * held[instrument.symbol]
            rows.append(_raw_row(sequence, instrument, day, "DISTRIBUTION", 0, 0, 0, amount))
        elif roll < 0.52:
            quantity = rng.choice((2, 3, 5, 7))
            rows.append(_raw_row(sequence, instrument, day, "BUY", quantity, price, 0, 0))
            held[instrument.symbol] += quantity
        elif held[instrument.symbol] > 2:
            quantity = min(rng.choice((1, 2, 3, 4)), held[instrument.symbol] - 1)
            fees = Decimal("4.95") if rng.random() < 0.25 else Decimal("0")
            rows.append(_raw_row(sequence, instrument, day, "SELL", quantity, price, fees, 0))
            held[instrument.symbol] -= quantity
        else:
            quantity = rng.choice((2, 3, 5))
            rows.append(_raw_row(sequence, instrument, day, "BUY", quantity, price, 0, 0))
            held[instrument.symbol] += quantity
        sequence += 1
    rows.sort(key=lambda row: (row["date"], int(row["sequence"])))
    return rows


def generate_transactions(config: GeneratorConfig = GeneratorConfig(),
                           instruments: Sequence[MarketInstrument] | None = None) -> list[dict[str, str]]:
    """Generate valid transactions using online-discovered instruments by default."""

    config.validate()
    selected = list(instruments) if instruments is not None else discover_instruments(config.instrument_count, config.seed)
    selected = _sample_instruments(selected, config.instrument_count, config.seed)
    return [_shape_row(row, config.shape) for row in _transactions(config, selected)]


def serialize_transactions(rows: list[dict[str, str]]) -> str:
    if not rows:
        return ""
    from io import StringIO

    output = StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=list(rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return output.getvalue()


def choose_output_path(shape: InputShape) -> Path | None:
    root = tk.Tk()
    root.withdraw()
    try:
        selected = filedialog.asksaveasfilename(
            title=f"Save Portfolio Compass {shape} sample",
            defaultextension=".csv",
            initialfile=f"portfolio_compass_{shape.replace('-', '_')}.csv",
            filetypes=[("CSV files", "*.csv")],
            confirmoverwrite=True,
        )
    finally:
        root.destroy()
    return Path(selected) if selected else None


def save_transactions(rows: list[dict[str, str]], destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(serialize_transactions(rows), encoding="utf-8", newline="")
    return destination


def main() -> None:
    parser = argparse.ArgumentParser(description="Generate synthetic stock/ETF transaction CSV input for Portfolio Compass.")
    parser.add_argument("--count", type=int, default=None, help="number of transactions (3-10,000); otherwise prompt")
    parser.add_argument("--instruments", type=int, default=None, help="number of stocks/investments (1-25); otherwise prompt")
    parser.add_argument("--shape", choices=SHAPES, default="canonical")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--no-distributions", action="store_true")
    parser.add_argument("--offline", action="store_true", help="use the broader built-in fallback universe")
    parser.add_argument("--output", type=Path, help="output CSV path; omit to open Save As")
    args = parser.parse_args()
    instrument_count = args.instruments if args.instruments is not None else _ask_count(
        "Number of Stocks/Investments", 3, 1, 25)
    transaction_count = args.count if args.count is not None else _ask_count(
        "Number of transactions (total)", 24, instrument_count, 10_000)
    config = GeneratorConfig(transaction_count, instrument_count, args.shape, args.seed, not args.no_distributions)
    instruments = _sample_instruments(OFFLINE_FALLBACK, instrument_count, args.seed) if args.offline else None
    source = "offline fallback" if args.offline else "Yahoo Finance"
    if instruments is None:
        try:
            instruments = discover_instruments(instrument_count, args.seed)
        except RuntimeError as exc:
            print(f"Online discovery unavailable ({exc}); using the labeled offline fallback.")
            instruments = _sample_instruments(OFFLINE_FALLBACK, instrument_count, args.seed)
            source = "offline fallback"
    rows = generate_transactions(config, instruments)
    destination = args.output or choose_output_path(config.shape)
    if destination is None:
        print("Save cancelled; no file was written.")
        return
    save_transactions(rows, destination)
    print(f"Saved {len(rows):,} synthetic transactions from {source} to {destination}")


def _ask_count(label: str, default: int, minimum: int, maximum: int) -> int:
    while True:
        raw = input(f"{label} [{default}]: ").strip()
        if not raw:
            return default
        try:
            value = int(raw)
        except ValueError:
            value = 0
        if minimum <= value <= maximum:
            return value
        print(f"Enter a whole number from {minimum:,} to {maximum:,}.")


if __name__ == "__main__":
    main()

```

### `portfolio_compass/ui.py`

This is the implementation snapshot used by the application.

```python
"""Streamlit presentation. Portfolio state stays inside the current session."""

import hashlib
import json
import time
from dataclasses import asdict
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

from .accounting import analyze
from .ai import GLOSSARY, answer, money
from .config import ai_available
from .importing import ALIASES, export_csv, normalize, read_csv, suggest_mapping
from .instruments import CATALOG, resolve_batch
from .market_data import demo_market, fetch_market, last_completed_session, lookup, session_dates
from .performance import comparison, period_metrics

ROOT = Path(__file__).resolve().parents[1]
PAGES = ['Get started', 'My portfolio', 'Performance', 'Ask Compass']


def clear_portfolio():
    for name in ['analysis', 'chat', 'last_symbol', 'summary', 'pending', 'transactions', 'resolved_instruments']:
        st.session_state.pop(name, None)


def reset():
    for name in list(st.session_state):
        del st.session_state[name]


def load_demo():
    clear_portfolio()
    headers, rows = read_csv((ROOT / 'Sample Data - Sheet1.csv').read_bytes())
    resolved = {item.symbol: item for item in CATALOG if item.symbol != 'SPY'}
    transactions, issues = normalize(rows, suggest_mapping(headers), resolved)
    market = demo_market()
    result = analyze(transactions, resolved, market, session_dates(transactions[0].day, market.as_of))
    result.issues.extend(issues)
    st.session_state.analysis = result
    st.session_state.page = 'My portfolio'


def theme():
    dark = st.session_state.get('theme') == 'Dark'
    bg, surface, text, muted, border = ('#0E1726', '#182538', '#F0F5FA', '#B9C9DB', '#35475D') if dark else ('#F5F7FA', '#FFFFFF', '#172B3A', '#53697B', '#DCE5EB')
    st.markdown(f'''<style>
    .stApp {{ background:{bg}; color:{text}; }}
    [data-testid="stSidebar"] {{ background:{surface}; border-right:1px solid {border}; }}
    [data-testid="stSidebar"] * {{ color:{text}; }}
    .block-container {{ max-width:1280px; padding-top:2rem; padding-bottom:3rem; }}
    h1,h2,h3,p,label {{ color:{text}; }}
    [data-testid="stMetric"] {{ background:{surface}; border:1px solid {border}; border-radius:16px; padding:20px; }}
    [data-testid="stMetricValue"] {{ color:{text}; font-weight:650; }}
    [data-testid="stCaptionContainer"] p {{ color:{muted}; }}
    [data-testid="stVerticalBlockBorderWrapper"] {{ border-radius:16px; }}
    .compass-brand {{ font-size:22px; font-weight:750; letter-spacing:-.6px; color:{text}; }}
    .compass-kicker {{ color:#25A8A0; letter-spacing:2px; font-size:12px; font-weight:700; margin-bottom:8px; }}
    .compass-hero {{ background:{surface}; border:1px solid {border}; border-radius:20px; padding:30px; margin-bottom:24px; }}
    .stButton button[kind="primary"] {{ background:#087F8C; border-color:#087F8C; color:white; }}
    @media(max-width:700px) {{ .block-container {{padding-left:1rem;padding-right:1rem;}} }}
    </style>''', unsafe_allow_html=True)  # Only application-authored constants; no user/model interpolation.
    return dark


def chart(fig, dark):
    fig.update_layout(template='plotly_dark' if dark else 'plotly_white', paper_bgcolor='rgba(0,0,0,0)',
                      plot_bgcolor='rgba(0,0,0,0)', margin=dict(l=12, r=12, t=25, b=16),
                      font=dict(family='Arial', size=13), legend=dict(orientation='h', y=-.2))
    st.plotly_chart(fig, width='stretch', config={'displaylogo': False})


def issues_view(issues):
    if not issues:
        return
    frame = pd.DataFrame([asdict(i) for i in issues])
    with st.expander(f'Data quality & assumptions · {len(issues)} notices'):
        st.dataframe(frame, hide_index=True, width='stretch')
        st.download_button('Download validation report', export_csv(frame.to_dict('records')), 'validation.csv', 'text/csv', key='issue-download')


def new_signature(signature):
    if st.session_state.get('import_signature') != signature:
        clear_portfolio()
        st.session_state.import_signature = signature


def get_started():
    st.markdown('<div class="compass-kicker">YOUR MONEY, MADE UNDERSTANDABLE</div>', unsafe_allow_html=True)
    st.title('A clearer view of your investments.')
    st.write('Turn a transaction spreadsheet into holdings, returns, and explanations you can follow.')
    left, right = st.columns([1, 1])
    with left, st.container(border=True):
        st.subheader('Take a guided first look')
        st.write('Explore Apple, Microsoft, and Tesla with a ready-to-use sample portfolio.')
        st.caption('Synthetic prices and distributions. Works offline. No account or API key needed.')
        st.button('Explore sample portfolio', type='primary', on_click=load_demo, width='stretch')
    with right, st.container(border=True):
        st.subheader('Bring your own history')
        st.write('Use tickers or common names. Codes and clear matches resolve automatically; only ambiguous names need your input.')
        st.download_button('Download CSV template', b'company,date,transaction_type,quantity,price,fees,cash_amount\nApple,2024-01-03,BUY,10,185.07,0,\n', 'portfolio-template.csv', 'text/csv', width='stretch')
        st.caption('USD · US investments · purchases, sales, distributions · complete history')
    st.divider()
    st.subheader('Upload your portfolio')
    st.caption('One CSV, up to 10 MB / 10,000 rows / 100 investments. Uploaded portfolios stay in this session.')
    upload = st.file_uploader('Transaction CSV', type=['csv'], key='upload')
    if upload is None:
        return
    data = upload.getvalue()
    fingerprint = hashlib.sha256(data).hexdigest()[:16]
    if st.session_state.get('upload_fingerprint') != fingerprint:
        clear_portfolio()
        st.session_state.upload_fingerprint = fingerprint
        st.session_state.lookup_results = {}
    with st.expander('File format', expanded=False):
        encoding = st.selectbox('Encoding', ['utf-8-sig', 'cp1252'])
        delimiter = st.selectbox('Delimiter', ['Auto', ',', ';', '\t', '|'])
        date_format = st.selectbox('Date format', ['ISO (YYYY-MM-DD)', 'US (MM/DD/YYYY)', 'Day first (DD/MM/YYYY)'])
        convention = st.selectbox('Number format', ['US', 'European'])
    try:
        headers, rows = read_csv(data, encoding, delimiter)
    except ValueError as exc:
        clear_portfolio()
        st.error(str(exc))
        return
    suggested = suggest_mapping(headers)
    st.markdown('**1 · Match your columns**')
    st.caption('Optional fields can remain unmapped. A name column can cross-check a separate ticker column.')
    mapping = {}
    columns = st.columns(3)
    for index, field in enumerate(ALIASES):
        options = ['— Not mapped —', *headers]
        default = options.index(suggested[field]) if suggested[field] else 0
        with columns[index % 3]:
            selected = st.selectbox(field.replace('_', ' ').title(), options, index=default, key=f'map-{fingerprint}-{field}')
            mapping[field] = '' if selected == options[0] else selected
    if any(not mapping[k] for k in ['instrument', 'date', 'type']):
        clear_portfolio()
        st.info('Map the investment, date and type fields to continue.')
        return
    raw_names = sorted({row[mapping[field]] for row in rows for field in ['instrument', 'name'] if mapping[field] and row[mapping[field]]})
    if len(raw_names) > 200:
        st.error('Use at most 200 unique raw names/tickers, resolving to at most 100 investments.')
        return
    context = hashlib.sha256(json.dumps([fingerprint, mapping, date_format, convention], sort_keys=True).encode()).hexdigest()
    if st.session_state.get('validation_context') != context:
        clear_portfolio()
        st.session_state.validation_context = context
        st.session_state.pop('identity_batch', None)
    resolved = validate_investments(raw_names, context)
    if resolved is None:
        return
    signature = hashlib.sha256(json.dumps([fingerprint, mapping, date_format, convention, {k: v.symbol for k, v in resolved.items()}], sort_keys=True).encode()).hexdigest()
    new_signature(signature)
    st.markdown('**3 · Review and analyze**')
    keep_duplicates = st.checkbox('Identical transaction rows are intentional (keep all of them)')
    transactions, issues = normalize(rows, mapping, resolved, date_format, convention, keep_duplicates)
    issues_view(issues)
    with st.expander('Preview normalized transactions', expanded=True):
        st.dataframe(pd.DataFrame([asdict(t) for t in transactions]).head(100).astype(str), hide_index=True, width='stretch')
    estimate = st.checkbox('Estimate missing distributions from market data', value=True)
    original_units = st.checkbox('Quantities and prices are original trade-date units (not retroactively split-adjusted).')
    confirmed = st.checkbox('I reviewed the mappings and warnings; this is my complete investment history.')
    errors = any(i.severity == 'error' for i in issues)
    if st.button('Analyze portfolio', type='primary', disabled=errors or not transactions or not original_units or not confirmed):
        clear_portfolio()
        instruments = {t.symbol: resolved[t.original] for t in transactions}
        try:
            with st.spinner('Fetching daily prices and reconciling your investment history…'):
                end = last_completed_session()
                market = fetch_market(list(instruments), transactions[0].day, end)
                if not estimate:
                    market.income_complete = False
                result = analyze(transactions, instruments, market, session_dates(transactions[0].day, end), estimate=estimate)
                result.issues.extend(issues)
            if result.conflicts:
                st.session_state.pending = (result, estimate)
            else:
                st.session_state.analysis = result
                st.success('Portfolio ready. Open My portfolio or Performance in the sidebar.')
        except ValueError as exc:
            st.error(str(exc))
    if 'pending' in st.session_state:
        income_review()


def validate_investments(raw_names, context):
    """Single validation action; only ambiguous names produce selection controls."""
    st.markdown('**2 · Validate investments**')
    st.caption('Validate all inputs together. Verified codes and clear names are accepted automatically.')
    if st.button('Validate inputs', type='primary'):
        clear_portfolio()
        for name in list(st.session_state):
            if name.startswith('ambiguous-'):
                del st.session_state[name]
        with st.spinner('Validating all investment codes and names…'):
            st.session_state.identity_batch = resolve_batch(raw_names, lookup)
    batch = st.session_state.get('identity_batch')
    if batch is None:
        st.info('Click Validate inputs once to check every investment in the file.')
        return None
    automatic, ambiguous, errors = batch
    resolved = dict(automatic)
    if ambiguous:
        st.warning(f'{len(ambiguous)} ambiguous name(s) need a selection. All are listed below.')
        for raw, candidates in ambiguous.items():
            labels = {f'{item.label} · {item.exchange} · {item.kind}': item for item in candidates}
            widget_key = 'ambiguous-' + hashlib.sha256((context + raw).encode()).hexdigest()[:16]
            selected = st.selectbox(f'Choose the investment for {raw}', list(labels), index=None, key=widget_key)
            if selected:
                resolved[raw] = labels[selected]
    for raw, message in errors.items():
        st.error(f'{raw}: {message}')
    if resolved:
        with st.expander(f'{len(resolved)} resolved inputs · view matches'):
            st.dataframe([{'Input': raw, 'Code': item.symbol, 'Investment': item.name,
                           'Status': 'Automatic' if raw in automatic else 'Selected'} for raw, item in resolved.items()],
                         hide_index=True, width='stretch')
    if errors or len(resolved) != len(raw_names):
        clear_portfolio()
        return None
    st.success('All investments resolved. No further identity confirmation is needed.')
    return resolved


def income_review():
    result, estimate = st.session_state.pending
    st.warning('Some uploaded distributions may overlap with online estimates. Resolve them before viewing results.')
    decisions = {}
    for conflict in result.conflicts:
        options = {'Separate income; keep provider estimates': 'independent'}
        options.update({f"Replace estimated event on {e['date']} ({money(float(e['amount']))})": e['id'] for e in conflict['candidates']})
        selection = st.selectbox(f"Row {conflict['row']} · {conflict['symbol']} · {conflict['date']} · {money(conflict['amount'])}", list(options), index=None, key=f"income-{conflict['id']}")
        if selection:
            decisions[conflict['id']] = options[selection]
    if st.button('Confirm income reconciliation', disabled=len(decisions) != len(result.conflicts)):
        try:
            result = analyze(result.transactions, result.instruments, result.market,
                             session_dates(result.transactions[0].day, result.market.as_of), decisions, estimate)
            st.session_state.analysis = result
            st.session_state.pop('pending')
            st.session_state.chat = []
            st.success('Income reconciled. Your portfolio is ready in the sidebar.')
        except ValueError as exc:
            st.error(str(exc))


def status(result):
    st.caption(f"{'SYNTHETIC DEMO' if result.market.demo else 'DAILY MARKET PRICES'} · USD · As of {result.market.as_of} · FIFO · Investment-only accounting")
    if not result.complete:
        st.warning('Valuation is incomplete. Priced subtotals exclude missing investments; complete portfolio metrics are unavailable.')
    if not result.market.income_complete:
        st.warning('Income coverage is incomplete. Results exclude unrecorded income; total-return benchmark comparison is unavailable.')


def overview(result, dark):
    st.title('Your portfolio, at a glance')
    status(result)
    end = result.daily[-1]
    metrics = period_metrics(result, result.daily[0]['date'], end['date'])
    cols = st.columns(4)
    cols[0].metric('Current investment value', money(end['value']), help='Value of the units still held. Brokerage cash is excluded.')
    cols[1].metric('Total profit / loss', money(end['profit']), help='Value + net sales + income − purchases including fees.')
    cols[2].metric('Annualized return · XIRR', f"{metrics['xirr']:.2%}" if metrics['xirr'] is not None else 'Unavailable', help=metrics['reason'])
    cols[3].metric('Income · recorded + estimated', money(end['income']), help=GLOSSARY['income'])
    if metrics['xirr'] is None:
        st.caption('XIRR: ' + metrics['reason'])
    left, right = st.columns([1.65, 1])
    with left, st.container(border=True):
        st.subheader('Your investment journey')
        frame = pd.DataFrame(result.daily)
        fig = px.line(frame, x='date', y='value', color_discrete_sequence=['#159B93'], labels={'date': '', 'value': 'Investment value (USD)'})
        fig.update_traces(connectgaps=False, fill='tozeroy', fillcolor='rgba(21,155,147,.08)')
        chart(fig, dark)
    with right, st.container(border=True):
        st.subheader('Where your money is')
        dimension = st.selectbox('Allocation view', ['Investment', 'Asset type', 'Sector'])
        active = pd.DataFrame([r for r in result.holdings if r['quantity'] > 0 and r['value'] is not None])
        if result.complete and not active.empty and active['value'].sum() > 0:
            field = {'Investment': 'investment', 'Asset type': 'type', 'Sector': 'sector'}[dimension]
            fig = px.pie(active, values='value', names=field, hole=.67,
                         color_discrete_sequence=['#159B93', '#407FB8', '#E5A54B', '#9B8ACB', '#D7767C'])
            chart(fig, dark)
            largest = active.loc[active['value'].idxmax()]
            st.caption(f"Largest holding: {largest['investment']} · {largest['value'] / end['value']:.1%}")
        else:
            st.info('Allocation requires complete prices and positive remaining holdings.')
    st.subheader('Holdings & gains')
    show_closed = st.checkbox('Include fully sold investments', value=True)
    frame = pd.DataFrame([r for r in result.holdings if show_closed or r['quantity'] > 0])
    st.dataframe(frame, hide_index=True, width='stretch', column_config={k: st.column_config.NumberColumn(k.replace('_', ' ').title(), format='$%.2f') for k in ['value', 'cost', 'price', 'realized', 'unrealized', 'income', 'profit']})
    st.download_button('Download holdings', export_csv(frame.to_dict('records')), 'holdings.csv', 'text/csv')
    with st.container(border=True):
        st.subheader('Compass brief')
        st.caption('Application-verified facts. Enable Groq on Ask Compass to let AI select the most relevant evidence.')
        if st.button('Generate portfolio summary'):
            st.session_state.summary = answer('Summarize my portfolio', result, st.session_state.get('ai_consent', False))
        brief = st.session_state.get('summary') or answer('Summarize my portfolio', result)
        st.text(brief['text'])
        st.caption(brief['method'])
    issues_view(result.issues)


def performance_view(result, dark):
    st.title('See the whole journey')
    status(result)
    selection = st.date_input('Reporting period', value=(result.daily[0]['date'], result.market.as_of), min_value=result.daily[0]['date'], max_value=result.market.as_of)
    if len(selection) != 2:
        st.info('Select a start and end date.')
        return
    start, end = selection
    metrics = period_metrics(result, start, end)
    cols = st.columns(4)
    for column, label, value in zip(cols, ['Purchases', 'Net sale proceeds', 'Income', 'Period profit'], [metrics.get('purchases'), metrics.get('sales'), metrics.get('income'), metrics['profit']]):
        column.metric(label, money(value))
    rate = metrics['xirr']
    st.metric('Annualized return · XIRR', f'{rate:.2%}' if rate is not None else 'Unavailable',
              help='Return for the selected reporting period, using dated cash flows and opening/closing investment values. ' + metrics['reason'])
    st.caption(metrics['reason'])
    st.subheader('Performance, on the same scale')
    benchmark = st.checkbox('Compare with S&P 500 (SPY proxy)', value=False)
    series, reason = comparison(result, start, end)
    if series:
        frame = pd.DataFrame(series)
        cols = ['Portfolio', 'S&P 500 (SPY proxy)'] if benchmark else ['Portfolio']
        chart(px.line(frame, x='date', y=cols, color_discrete_sequence=['#159B93', '#E5A54B'], labels={'date': '', 'value': 'Cumulative return (%)', 'variable': ''}), dark)
    else:
        st.info(reason)
    st.caption('Daily approximation: purchases at the start of the day, sales and income at the end. Empty-capital periods and missing data break comparisons.')
    if series:
        st.caption(reason)
    st.subheader('Value and cumulative money flows')
    history = pd.DataFrame(metrics['rows'])
    chosen = st.multiselect('Show series', ['value', 'purchases', 'sale_proceeds', 'income', 'profit'], default=['value', 'purchases', 'sale_proceeds'])
    if chosen:
        chart(px.line(history, x='date', y=chosen, labels={'date': '', 'value': 'USD', 'variable': ''}), dark)
    st.caption('Cumulative flow series run from the first transaction; period cards above use only the selected dates.')
    with st.expander('Accessible chart data & downloads'):
        st.dataframe(history, hide_index=True, width='stretch')
        st.download_button('Download performance', export_csv(metrics['rows']), 'performance.csv', 'text/csv')
        st.download_button('Download normalized transactions', export_csv([asdict(t) for t in result.transactions]), 'transactions.csv', 'text/csv')
    with st.expander('Income ledger · recorded and estimated'):
        records = [{**r, 'amount': float(r['amount']), 'units': float(r['units']) if r['units'] is not None else None} for r in result.income]
        st.dataframe(pd.DataFrame(records), hide_index=True, width='stretch')
        st.download_button('Download income ledger', export_csv(records), 'income.csv', 'text/csv')
    with st.expander('How these numbers work'):
        for name, definition in GLOSSARY.items():
            if name != 'upload':
                st.write(f'**{name.upper()}**')
                st.write(definition)


def chat_view(result):
    st.title('Ask Compass')
    st.write('Explore your holdings and performance in plain language.')
    status(result)
    st.info('Scope: your uploaded investments and the S&P 500 comparison only. No trade recommendations, forecasts, or unrelated answers.')
    available = ai_available()
    st.checkbox('Enable Groq: send only relevant portfolio facts to the AI service', key='ai_consent', disabled=not available)
    st.caption('No raw CSV or API key is sent in prompts. Responses display verified fact templates only. ' + ('Groq is configured.' if available else 'No Groq key configured. Local factual answers are available.'))
    suggestions = ['Summarize my portfolio', 'Which holding contributed the most profit?', 'How concentrated is my portfolio?', 'Compare my return with the S&P 500', 'Explain XIRR']
    suggested = st.selectbox('Try a question', ['Choose a question…', *suggestions])
    question = None
    if st.button('Ask selected question', disabled=suggested == 'Choose a question…'):
        question = suggested
    history = st.session_state.setdefault('chat', [])
    for entry in history:
        with st.chat_message(entry['role']):
            st.text(entry['text'])
            if entry.get('method'):
                st.caption(entry['method'])
    typed = st.chat_input('Ask about an investment, a metric, or your portfolio', max_chars=1000)
    question = typed or question
    if question:
        now = time.monotonic()
        recent = [t for t in st.session_state.get('chat_times', []) if now - t < 60]
        if len(recent) >= 10:
            st.warning('Please wait a minute before asking more questions.')
            return
        st.session_state.chat_times = [*recent, now]
        response = answer(question, result, bool(st.session_state.get('ai_consent')) and available, st.session_state.get('last_symbol'))
        st.session_state.last_symbol = response['symbol']
        history.extend([{'role': 'user', 'text': question}, {'role': 'assistant', **response}])
        st.session_state.chat = history[-20:]
        st.rerun()
    if history and history[-1].get('evidence'):
        with st.expander('Evidence behind the latest answer'):
            st.json(history[-1]['evidence'])
    st.button('Clear this conversation', on_click=lambda: [st.session_state.pop(k, None) for k in ['chat', 'last_symbol']])


def main():
    st.set_page_config(page_title='Portfolio Compass', page_icon='🧭', layout='wide')
    with st.sidebar:
        st.markdown('<div class="compass-brand">◈ Portfolio Compass</div>', unsafe_allow_html=True)
        st.caption('Understand your investments.')
        st.divider()
        page = st.radio('Navigate', PAGES, key='page', label_visibility='collapsed')
        st.divider()
        st.selectbox('Appearance', ['Light', 'Dark'], key='theme')
        st.caption('USD · US markets · FIFO')
        st.button('Reset portfolio & session', on_click=reset, width='stretch')
        st.caption('Your uploads are not saved to a database. Re-upload after a new session.')
    dark = theme()
    result = st.session_state.get('analysis')
    if page == 'Get started':
        get_started()
    elif result is None:
        st.title('Your portfolio starts here')
        st.info('Upload your transactions on Get started, or explore the sample.')
        st.button('Explore sample portfolio', type='primary', on_click=load_demo)
    elif page == 'My portfolio':
        overview(result, dark)
    elif page == 'Performance':
        performance_view(result, dark)
    else:
        chat_view(result)
    st.divider()
    st.caption('Portfolio Compass · Clear numbers. Visible assumptions. · Educational portfolio analytics, not trading recommendations.')

```

### `sample_data_generator.py`

This is the implementation snapshot used by the application.

```python
"""Convenience launcher for the standalone Sample Data Generator."""

from __future__ import annotations

import importlib.util
import os
from pathlib import Path
import sys


def _use_project_environment() -> None:
    """Re-launch with the project venv when the shell selected another Python."""

    venv_python = Path(__file__).resolve().parent / ".venv" / "Scripts" / "python.exe"
    current_python = Path(sys.executable).resolve()
    if venv_python.exists() and current_python != venv_python and importlib.util.find_spec("yfinance") is None:
        os.execv(str(venv_python), [str(venv_python), __file__, *sys.argv[1:]])


if __name__ == "__main__":
    _use_project_environment()
    from portfolio_compass.sample_data_generator import main

    main()

```